In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('../data/players_clean.csv')

In [3]:
df['goals_per90']        = (df['total_goals']       / df['total_minutes']) * 90
df['assists_per90']      = (df['total_assists']     / df['total_minutes']) * 90
df['goal_contributions'] = df['total_goals'] + df['total_assists']
df['gc_per90']           = (df['goal_contributions']/ df['total_minutes']) * 90

df.replace([np.inf, -np.inf], np.nan, inplace=True)
df[['goals_per90','assists_per90','gc_per90']] = \
    df[['goals_per90','assists_per90','gc_per90']].fillna(0)

df[['name','position','sub_position',
    'goals_per90','assists_per90','gc_per90']]\
  .sort_values('gc_per90', ascending=False).head(10)

,name,position,sub_position,goals_per90,assists_per90,gc_per90
2421,Dusan Predrag Djuric,Midfield,Central Midfield,0.937500,1.875000,2.812500
23086,Osman Addo,Attack,Left Winger,2.195122,0.439024,2.634146
13073,Carlo de Reuver,Attack,Centre-Forward,2.195122,0.000000,2.195122
22992,Hilmir Rafn Mikaelsson,Attack,Centre-Forward,2.000000,0.000000,2.000000
25086,Bachir Gueye,Attack,Centre-Forward,2.000000,0.000000,2.000000
20674,Arman Taranis,Attack,Centre-Forward,0.957447,0.957447,1.914894
21704,Özgür Sert,Midfield,Central Midfield,0.957447,0.957447,1.914894
15939,Ferdinando Del Sole,Attack,Right Winger,1.862069,0.000000,1.862069
23202,Steven Nsimba,Attack,Centre-Forward,1.836735,0.000000,1.836735
14691,Issa Baradji,Attack,Centre-Forward,1.800000,0.000000,1.800000


In [4]:
df['yellow_per90'] = (df['total_yellow_cards'] / df['total_minutes']) * 90
df['red_per90']    = (df['total_red_cards']    / df['total_minutes']) * 90

df.replace([np.inf, -np.inf], np.nan, inplace=True)
df[['yellow_per90','red_per90']] = \
    df[['yellow_per90','red_per90']].fillna(0)

print("Avg yellow cards per 90 by position:")
print(df.groupby('position')['yellow_per90'].mean().round(3).sort_values(ascending=False))

Avg yellow cards per 90 by position:
position
Midfield      0.247
Missing       0.237
Defender      0.221
Attack        0.196
Goalkeeper    0.060
Name: yellow_per90, dtype: float64


In [5]:
df['minutes_ratio'] = df['total_minutes'] / df['total_minutes'].max()

df['market_value_log'] = np.log1p(df['market_value_in_eur'])

print(df[['name','market_value_in_eur','market_value_log']].head(5))

                 name  market_value_in_eur  market_value_log
0      Miroslav Klose            1000000.0         13.815512
1  Roman Weidenfeller             750000.0         13.527830
2    Dimitar Berbatov            1000000.0         13.815512
3               Lúcio             200000.0         12.206078
4          Tom Starke             100000.0         11.512935


In [6]:
position_dummies = pd.get_dummies(df['position'], prefix='pos')
df = pd.concat([df, position_dummies], axis=1)

print("New position columns:")
print([c for c in df.columns if c.startswith('pos_')])

New position columns:
['pos_Attack', 'pos_Defender', 'pos_Goalkeeper', 'pos_Midfield', 'pos_Missing']


In [7]:
print(df[['name','position','pos_Attack','pos_Midfield',
         'pos_Defender','pos_Goalkeeper']].head(6))

                 name    position  pos_Attack  pos_Midfield  pos_Defender  \
0      Miroslav Klose      Attack        True         False         False   
1  Roman Weidenfeller  Goalkeeper       False         False         False   
2    Dimitar Berbatov      Attack        True         False         False   
3               Lúcio    Defender       False         False          True   
4          Tom Starke  Goalkeeper       False         False         False   
5                Dedê    Defender       False         False          True   

   pos_Goalkeeper  
0           False  
1            True  
2           False  
3           False  
4            True  
5           False  


In [8]:
for col in ['goals_per90','assists_per90','gc_per90','yellow_per90']:
    cap = df[col].quantile(0.99)
    df[col] = df[col].clip(upper=cap)

CLUSTER_FEATURES = [
    'goals_per90',
    'assists_per90',
    'gc_per90',
    'yellow_per90',
    'minutes_ratio',
    'market_value_log',
    'pos_Attack',
    'pos_Midfield',
    'pos_Defender',
    'pos_Goalkeeper',
]

df_features = df[CLUSTER_FEATURES].fillna(0)

print(f"Feature matrix shape: {df_features.shape}")
print(df_features.describe().round(3))

Feature matrix shape: (25088, 10)
       goals_per90  assists_per90   gc_per90  yellow_per90  minutes_ratio  \
count    25088.000      25088.000  25088.000     25088.000      25088.000   
mean         0.108          0.082      0.193         0.205          0.095   
std          0.151          0.105      0.216         0.179          0.124   
min          0.000          0.000      0.000         0.000          0.002   
25%          0.000          0.000      0.000         0.080          0.011   
50%          0.042          0.045      0.116         0.180          0.044   
75%          0.161          0.131      0.312         0.282          0.130   
max          0.709          0.528      0.938         0.938          1.000   

       market_value_log  
count         25088.000  
mean             12.764  
std               2.441  
min               0.000  
25%              11.918  
50%              12.692  
75%              13.816  
max              19.114  


In [9]:
df.to_csv('../data/players_features.csv', index=False)
print("Saved players_features.csv — ready for Phase 5")

Saved players_features.csv — ready for Phase 5
